---
title: "Building Intelligent Customer Service Agents with Pydantic-AI and LLMs
"
author: "Fabian"
date: "2025-01-19"
categories: [Agents]
format:
  html:
    code-fold: false
    toc: true
    echo: true
jupyter: python3
---

## Introduction: Unleashing the Power of LLMs for Customer Service


Man I was blown away by how simple and effective building agents in pydantic has been for me this weekend. 

The promise of AI-powered customer service is finally within reach. This tutorial demonstrates how to build an intelligent customer service agent using **Pydantic-AI**, a framework that elegantly marries the robustness of Pydantic's type safety with the sophisticated reasoning capabilities of Large Language Models (LLMs).

This tutorial will show you how to build an intelligent customer service agent using Pydantic-AI, a powerful framework that combines the type safety of Pydantic with the capabilities of Large Language Models (LLMs).

Tired of brittle, hard-coded bots?  I'll guide you through building a complete customer service solution capable of:

*   **Looking up customer orders:** Quickly retrieve order details based on customer ID.
*   **Handling support tickets:**  Access and manage customer support requests efficiently.
*   **Accessing customer information:** Securely retrieve customer profiles.
*   **Processing returns and refunds:** Automate return and refund processes.

## 1. Setting Up the Development Environment

First, let's set up our development environment with the necessary dependencies:
```bash
pip install pydantic-ai
```

We'll also need to install and run Ollama locally for our LLM backend. 
just use brew install ollama and then download your model of choice.


In [1]:
import os
from dotenv import load_dotenv
from pydantic_ai import Agent, RunContext, ModelRetry
from pydantic_ai.models.openai import OpenAIModel  # Changed import
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Literal
from datetime import date
from dataclasses import dataclass
from pydantic import ValidationError

import nest_asyncio
nest_asyncio.apply()

# Load environment variables from .env file
load_dotenv()

# Get the OpenAI API key from the environment variable
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")  

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY environment variable not set.")

# Initialize our LLM model
model = OpenAIModel( 
    model_name='gpt-4o-mini', 
    api_key=OPENAI_API_KEY,
)

## 2. Defining Our Data Models

Let's expand on your initial code to create a more comprehensive data model structure:

In [2]:

class Customer(BaseModel):
    id: int
    name: str
    email: str
    join_date: date

class OrderItem(BaseModel):
    item_id: int
    name: str
    quantity: int
    price: float

class Order(BaseModel):
    order_id: int
    customer_id: int
    items: List[OrderItem]
    total: float
    status: Literal['pending', 'shipped', 'delivered', 'returned']
    order_date: date

class SupportTicket(BaseModel):
    ticket_id: int
    customer_id: int
    order_id: int | None
    subject: str
    description: str
    status: Literal['open', 'in_progress', 'resolved', 'closed']
    priority: Literal['low', 'medium', 'high', 'urgent']
    created_date: date

### Why This Matters

- Type Safety: Pydantic validates data at runtime, catching errors early
- Auto-completion: Your IDE can provide suggestions as you code
- Documentation: The models serve as self-documenting code
- AI Understanding: The LLM can better understand the structure of your data

## 3. Implementing the Database Layer

Let's enhance your OrderDatabase class to handle our expanded data models:

In [10]:
class CustomerServiceDatabase:
    def __init__(self):
        # Simulated database tables
        self.customers: Dict[int, Customer] = {}
        self.orders: Dict[int, List[Order]] = {}
        self.tickets: Dict[int, List[SupportTicket]] = {}
        
        # Initialize with some sample data
        self._init_sample_data()
    
    async def get_customer(self, customer_id: int) -> Customer:
        if customer_id not in self.customers:
            raise ModelRetry(f"Customer not found: {customer_id}")
        return self.customers[customer_id]
    
    async def get_orders(self, customer_id: int) -> List[Order]:
        if customer_id not in self.orders:
            return []
        return self.orders[customer_id]
    
    async def get_tickets(self, customer_id: int) -> List[SupportTicket]:
        if customer_id not in self.tickets:
            return []
        return self.tickets[customer_id]
    
    async def create_ticket(self, ticket: SupportTicket) -> SupportTicket:
        if ticket.customer_id not in self.customers:
            raise ModelRetry("Invalid customer ID")
        
        customer_tickets = self.tickets.get(ticket.customer_id, [])
        customer_tickets.append(ticket)
        self.tickets[ticket.customer_id] = customer_tickets
        return ticket

    def _init_sample_data(self):
        # Add sample customers, orders, and tickets for testing
        self.customers[1] = Customer(id=1, name="Alice Smith", email="alice.smith@example.com", join_date=date(2024, 1, 15))
        self.orders[1] = [
            Order(order_id=101, customer_id=1, items=[OrderItem(item_id=201, name="Laptop", quantity=1, price=1200.00)], total=1200.00, status="shipped", order_date=date(2024, 11, 10)),
            Order(order_id=102, customer_id=1, items=[OrderItem(item_id=202, name="Keyboard", quantity=1, price=75.00)], total=75.00, status="delivered", order_date=date(2024, 11, 15))
        ]
        self.tickets[1] = [
            SupportTicket(ticket_id=301, customer_id=1, order_id=101, subject="Shipping Issue", description="Laptop never arrived", status="open", priority="high", created_date=date(2024, 12, 1))
        ]

## 4. Building the Agent Tools

Now let's create a more comprehensive agent with multiple tools:

In [4]:
@dataclass
class CustomerServiceDeps:
    customer_id: int
    db: CustomerServiceDatabase

class CustomerServiceResult(BaseModel):
    customer_id: int
    customer_info: Customer | None = None
    orders: List[Order] = Field(default_factory=list)
    tickets: List[SupportTicket] = Field(default_factory=list)
    message: str = Field(default="")



In [5]:
# Create the customer service agent
customer_service_agent = Agent(
    model=model,
    deps_type=CustomerServiceDeps,
    result_type=CustomerServiceResult,
    retries=2,
    system_prompt="""You are a customer service assistant with access to customer data, 
    orders, and support tickets. Help customers by providing accurate information and 
    creating support tickets when needed."""
)

In [6]:
# Define agent tools
@customer_service_agent.tool
async def get_customer_info(ctx: RunContext[CustomerServiceDeps]) -> Customer:
    """Fetch customer information based on their ID."""
    return await ctx.deps.db.get_customer(ctx.deps.customer_id)

@customer_service_agent.tool
async def get_customer_orders(ctx: RunContext[CustomerServiceDeps]) -> List[Order]:
    """Fetch all orders for a customer."""
    return await ctx.deps.db.get_orders(ctx.deps.customer_id)

@customer_service_agent.tool
async def get_support_tickets(ctx: RunContext[CustomerServiceDeps]) -> List[SupportTicket]:
    """Fetch all support tickets for a customer."""
    return await ctx.deps.db.get_tickets(ctx.deps.customer_id)

@customer_service_agent.tool
async def create_support_ticket(
    ctx: RunContext[CustomerServiceDeps],
    subject: str,
    description: str,
    priority: Literal['low', 'medium', 'high', 'urgent'],
    order_id: int | None = None
) -> SupportTicket:
    """Create a new support ticket for the customer."""
    ticket = SupportTicket(
        ticket_id=len(ctx.deps.db.tickets.get(ctx.deps.customer_id, [])) + 1,
        customer_id=ctx.deps.customer_id,
        order_id=order_id,
        subject=subject,
        description=description,
        status='open',
        priority=priority,
        created_date=date.today()
    )
    return await ctx.deps.db.create_ticket(ticket)

## 5. Using the Agent

Here's how to use our enhanced customer service agent:

In [11]:
async def handle_customer_inquiry(
    customer_id: int,
    inquiry: str,
    db: CustomerServiceDatabase
) -> CustomerServiceResult:
    deps = CustomerServiceDeps(customer_id=customer_id, db=db)
    
    try:
        result = await customer_service_agent.run(
            inquiry,
            deps=deps
        )
        return result.data # Access the data attribute
    except ValidationError as e:
        print(f"Validation error: {e}")
        raise
    except ModelRetry as e:
        print(f"Retryable error: {e}")
        raise

In [13]:
import asyncio

db = CustomerServiceDatabase()

# Example customer inquiry
inquiry = "I'd like to know the status of my recent orders and if there are any open support tickets."

async def run_inquiry():
    try:
        deps = CustomerServiceDeps(customer_id=1, db=db)  # Create dependencies

        result = await handle_customer_inquiry(
            customer_id=1,
            inquiry=inquiry,
            db=db
        )

        print(f"Customer Info: {result.customer_info}")
        print(f"Orders: {result.orders}")
        print(f"Tickets: {result.tickets}")
        print(f"Response: {result.message}")

    except Exception as e:
        print(f"Error handling inquiry: {e}")

# Run the asynchronous function
asyncio.run(run_inquiry())

Customer Info: id=1 name='Alice Smith' email='alice.smith@example.com' join_date=datetime.date(2024, 1, 15)
Orders: [Order(order_id=101, customer_id=1, items=[OrderItem(item_id=201, name='Laptop', quantity=1, price=1200.0)], total=1200.0, status='shipped', order_date=datetime.date(2024, 11, 10)), Order(order_id=102, customer_id=1, items=[OrderItem(item_id=202, name='Keyboard', quantity=1, price=75.0)], total=75.0, status='delivered', order_date=datetime.date(2024, 11, 15))]
Tickets: [SupportTicket(ticket_id=301, customer_id=1, order_id=101, subject='Shipping Issue', description='Laptop never arrived', status='open', priority='high', created_date=datetime.date(2024, 12, 1))]
Response: 



## 6. Error Handling and Validation

The agent includes built-in error handling for common scenarios:
- Invalid customer IDs
- Missing orders or tickets
- Validation errors in data models
- Retry logic for transient failures

## 7. Testing the Agent

Here's a basic test suite for our customer service agent:

In [14]:
import pytest
from datetime import date

@pytest.mark.asyncio
async def test_customer_lookup():
    db = CustomerServiceDatabase()
    # Add test customer
    db.customers[1] = Customer(
        id=1,
        name="Test Customer",
        email="test@example.com",
        join_date=date.today()
    )
    
    result = await handle_customer_inquiry(
        customer_id=1,
        inquiry="What is my customer information?",
        db=db
    )
    
    assert result.customer_info is not None
    assert result.customer_info.id == 1

@pytest.mark.asyncio
async def test_order_lookup():
    db = CustomerServiceDatabase()
    # Add test order
    db.orders[1] = [
        Order(
            order_id=1,
            customer_id=1,
            items=[
                OrderItem(
                    item_id=1,
                    name="Test Item",
                    quantity=1,
                    price=99.99
                )
            ],
            total=99.99,
            status="shipped",
            order_date=date.today()
        )
    ]
    
    result = await handle_customer_inquiry(
        customer_id=1,
        inquiry="What are my recent orders?",
        db=db
    )
    
    assert len(result.orders) > 0
    assert result.orders[0].order_id == 1